# SDXL 建筑场景两阶段生成

第一阶段用建筑遮罩保护主体，只生成场地环境；第二阶段使用 Canny 统一建筑、景观、铺装与光影。模型缓存和结果都写入 Google Drive。

## 1. 挂载 Drive 并设置持久化缓存

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ifc-ai-render')
CACHE_DIR = Path('/content/drive/MyDrive/ifc-ai-render-cache/huggingface')
RESULT_DIR = DRIVE_ROOT / 'results' / 'sdxl_scene_two_stage'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(CACHE_DIR / 'hub')
print('模型缓存:', CACHE_DIR)
print('实验结果:', RESULT_DIR)

## 2. 确认 GPU 并获取项目

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '请在 Colab 的运行时设置中启用 GPU'
print('当前 GPU:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess

REPO_URL = 'https://github.com/StrawberryChen/ifc-ai-render.git'
REPO_DIR = Path('/content/ifc-ai-render')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('当前代码版本:')
!git log -1 --oneline

## 3. 安装依赖（安装后若 Colab 提示重启，重启后从第1节重新运行）

In [ ]:
!pip -q install -r requirements-colab.txt

## 4. 查看并调整实验配置

请整段运行本单元。`scene_prompt` 负责场地内容，建筑遮罩保护主体；第二阶段只做低强度整体统一。

In [ ]:
import json

CONFIG_PATH = REPO_DIR / 'configs' / 'sdxl_scene_two_stage.json'
RUN_CONFIG_PATH = Path('/content/sdxl_scene_two_stage_run.json')
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

config['input']['image'] = str(REPO_DIR / 'outputs' / 'sdcc' / 'sdcc.material.png')
config['input']['building_mask'] = str(REPO_DIR / 'outputs' / 'sdcc' / 'sdcc.building_mask.png')
config['output']['directory'] = str(RESULT_DIR)
config['inpaint']['seed'] = 42
config['inpaint']['strength'] = 0.99
config['inpaint']['num_inference_steps'] = 30
config['inpaint']['guidance_scale'] = 7.0
config['input']['mask_dilation'] = 10
config['input']['mask_blur'] = 4
config['refinement']['overrides']['strengths'] = [0.30, 0.40]
config['refinement']['overrides']['conditioning_scale'] = 0.65
RUN_CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(RUN_CONFIG_PATH.read_text(encoding='utf-8'))

## 5. 先校验路径和配置，不下载模型

In [ ]:
!python inference/generate_sdxl_scene_two_stage.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR" \
  --validate-only

## 6. 运行场地生成与整体润色

第一次执行会新增下载 SDXL Inpainting 权重到同一个 Drive 缓存。两个阶段顺序加载模型并主动释放显存。

In [ ]:
!python inference/generate_sdxl_scene_two_stage.py \
  --config "$RUN_CONFIG_PATH" \
  --cache-dir "$CACHE_DIR"

## 7. 并排查看输入与结果

In [ ]:
from PIL import Image
from IPython.display import display

print('基础材质图')
display(Image.open(config['input']['image']))
print('允许重绘的环境遮罩（白色可生成，黑色保护建筑）')
display(Image.open(RESULT_DIR / 'input.environment_mask.png'))
print('第一阶段：场地环境生成')
display(Image.open(RESULT_DIR / 'stage1.environment.png'))
for result_path in sorted((RESULT_DIR / 'stage2_refined').glob('sdcc.strength_*.png')):
    print('第二阶段：', result_path.name)
    display(Image.open(result_path))
print((RESULT_DIR / 'two_stage_metadata.json').read_text(encoding='utf-8'))

## 8. 检查 Drive 中的模型和结果占用

In [ ]:
!du -sh "$CACHE_DIR" "$RESULT_DIR" 2>/dev/null || true
!find "$RESULT_DIR" -maxdepth 1 -type f -print